In [4]:
# TODO:
    # turn wkt into geom 
    #  use python module shapely?
    #  test to ingest into postgis docker when running 
    # write script to extract raw from postgis to transform 
import html
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
import polars as pl
import requests
import json
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

In [5]:


def extract_all_boroughs():
    """
    for each borough in the static list of boroughs with known subdomains:
    
    BASE_WFS is the endpoint for the web feature service that provides the traffic order data in GML format
    url is the URL for the webpage that contains the layer configuration for the traffic orders
    headers is a dictionary that sets the User-Agent header for the HTTP request to mimic a browser
    soup is a BeautifulSoup object that parses the HTML content of the response from the URL

    layer input is a list of dict objects that contains layer configuration information ie the layername and orderIDS
    defel is the key in the layer config that contains the orderIDs for that layer 

    load the layer into json and create a mapping of the layer name to the OrderIDs for that layer

    loop over the layer_id_map to get key value pairs of layer name and orderIDs

    skips layer name signs as it is not relevant to the traffic orders

    for each layer it makes a WTS request get the features for that layer 

    ET.fromstring is used to parse the GML response into an XML tree structure for further processing

    features are extracted from XML tree using correct path through GML featureMember elements

    loop over the features another for loop over feature to extract child elements 

    using the tag and split the tag to get actual tag name for cols heading 

    using the text of the child element as the value for that column in the record dict

    to get the geom look into msGeometry tag then into gml:coordinates as its nested 
    and extract the raw coordinates text

    create a raw geom col with raw coordinates 

    create a geom_srid col with the SRID for the coordinates

    create a wkt col by converting the raw coordinates into WKT format

    for geometry type detection:
        if 1 pair of coordinates it is a Point
        if 2 pairs of coordinates it is a LineString
        if more than 2 pairs of coordinates check if first and last coordinate are the same
            if they are the same it is a Polygon (closed ring)
            if they are not the same it is a LineString (open ring)

    append the record dict to all_features list which will be used to create a DataFrame at the end
    """

    
    STATIC_BOROUGHS = {
    "barking_dagenham": "barking-dagenham",
    "barnet":           "barnet",
    "camden":           "camden",
    "enfield":          "enfield",
    "hackney":          "hackney",
    "hounslow":         "hounslow",
    "lewisham":         "lewisham",
    "redbridge":        "redbridge",
}
    all_features = []
    
    for borough_key, subdomain in STATIC_BOROUGHS.items():
        BASE_WFS = f"https://mapserver.traffweb.app/cgi-bin/{subdomain}/parkmap"
        url = f"https://{subdomain}.traffweb.app/traffweb/1/TrafficOrders"
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

        print(f"\n=== {borough_key.upper()} ===")

        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        layer_input = soup.find("input", {"id": "layerconfig"})

        if not layer_input:
            print(f"  No layerconfig found — skipping")
            continue

        raw = html.unescape(layer_input.get("value", ""))
        layers = json.loads(raw)
        layer_id_map = {
            layer["name"]: layer["defsel"]
            for layer in layers
            if layer["defsel"]
        }

        for layer_name, order_ids in layer_id_map.items():
            if layer_name == "signs":
                print(f"  Skipping {layer_name}")
                continue

            print(f"  Fetching {layer_name}...")

            params = {
                "SERVICE": "WFS",
                "VERSION": "1.1.0",
                "REQUEST": "GetFeature",
                "TYPENAME": layer_name,
                "OUTPUTFORMAT": "GML2",
                "SRSNAME": "EPSG:27700",
                "MYORDERS": order_ids,
            }

            wfs_response = requests.get(BASE_WFS, params=params, timeout=120)

            if wfs_response.status_code != 200:
                print(f"    Skipping — status {wfs_response.status_code}")
                continue

            try:
                root = ET.fromstring(wfs_response.text)
                ns = {
                    "wfs": "http://www.opengis.net/wfs",
                    "gml": "http://www.opengis.net/gml",
                    "ms":  "http://mapserver.gis.umn.edu/mapserver",
                }

                features = root.findall(f"gml:featureMember/ms:{layer_name}", ns)
                print(f"    Features found: {len(features)}")

                for feature in features:
                    record = {"layer": layer_name, "borough": borough_key}

                    for child in feature:
                        tag = child.tag.split("}")[-1]
                        record[tag] = child.text

                        if tag == "msGeometry":
                            coords_elem = child.find(".//gml:coordinates", ns)
                            if coords_elem is not None and coords_elem.text:
                                raw = coords_elem.text.strip()
                                record["geom_raw"] = raw
                                record["geom_srid"] = 27700
                                pairs = raw.split()

                                if len(pairs) == 1:
                                    x, y = pairs[0].split(",")
                                    record["wkt"] = f"POINT ({x} {y})"
                                    record["geom_type"] = "Point"
                                elif len(pairs) == 2:
                                    wkt_coords = ", ".join(
                                        f"{p.split(',')[0]} {p.split(',')[1]}"
                                        for p in pairs
                                    )
                                    record["wkt"] = f"LINESTRING ({wkt_coords})"
                                    record["geom_type"] = "LineString"
                                elif len(pairs) > 2:
                                    wkt_coords = ", ".join(
                                        f"{p.split(',')[0]} {p.split(',')[1]}"
                                        for p in pairs
                                    )
                                    first, last = pairs[0], pairs[-1]
                                    if first == last:
                                        record["wkt"] = f"POLYGON (({wkt_coords}))"
                                        record["geom_type"] = "Polygon"
                                    else:
                                        record["wkt"] = f"LINESTRING ({wkt_coords})"
                                        record["geom_type"] = "LineString"
                            else:
                                record["geom_raw"] = None
                                record["geom_srid"] = None
                                record["wkt"] = None
                                record["geom_type"] = None

                    all_features.append(record)

            except ET.ParseError as e:
                print(f"    Parse error: {e}")

    return pl.DataFrame(all_features,infer_schema_length=None)


df_traffweb = extract_all_boroughs()
print(f"\nTotal features: {df_traffweb.shape}")
df_traffweb.head()


=== BARKING_DAGENHAM ===
  Fetching ordersr...
    Features found: 120
  Fetching ordersl...
    Features found: 31923
  Fetching mordersr...
    Features found: 19
  Fetching mordersl...
    Features found: 94
  Fetching mordersp...
    Features found: 1

=== BARNET ===
  Fetching ordersr...
    Features found: 310
  Fetching ordersl...
    Features found: 30529
  Fetching ordersp...
    Features found: 17242

=== CAMDEN ===
  Fetching twpzones...
    Features found: 50
  Fetching twtzones...
    Features found: 4
  Fetching ordersr...
    Features found: 254
  Fetching ordersl...
    Features found: 24510
  Fetching mordersl...
    Features found: 70
  Fetching mordersp...
    Features found: 6

=== ENFIELD ===
  Fetching twpzones...
    Features found: 31
  Fetching ordersr...
    Features found: 148
  Fetching ordersl...
    Features found: 20827

=== HACKNEY ===
  Fetching twtzones...
    Features found: 4
  Fetching ordersr...
    Features found: 157
  Fetching ordersl...
    Fe

layer,borough,boundedBy,msGeometry,geom_raw,geom_srid,wkt,geom_type,ogc_fid,item_ref,order_ref,order_type,street_name,side_of_road,locality,district,ordstart,ordfinish,ordlocation,schedule,date_from,date_to,pre_description,times_of_enforcement,post_description,sp_filename,sp_blockname,entry_type,restriction,pm_id,order_id,order_doc,blockimagefile,ms_grid,type_ref,no_of_spaces,length,tariff_code,tariff,pbp_code,pbp_tariff,zone_code,order_type_ln,side_of_road_ln,restriction_ln,tariff1,tariff2,poly_id,polyname,poly_info,zonecode,charges_doc
str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,null,str,null,str,null,str,null,str,str,str,str,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545286.070000,183948.350000 54…",27700,"""POLYGON ((545286.070000 183948…","""Polygon""","""1892753""","""239941""","""Consolidation 2016""","""Permit Parking Area""","""Lancaster Avenue""",null,null,"""Barking""",null,null,null,"""X640""","""2016-09-20""","""2199-01-01""",null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46956""","""1022""",null,null,""",1,""","""73""","""0""","""9.2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545246.110000,184107.900000 54…",27700,"""POLYGON ((545246.110000 184107…","""Polygon""","""1892754""","""239942""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""","""2016-09-20""","""2199-01-01""",null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46957""","""1022""",null,null,""",1,""","""73""","""0""","""0""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545235.900000,184109.950000 54…",27700,"""POLYGON ((545235.900000 184109…","""Polygon""","""1892755""","""239943""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""","""2016-09-20""","""2199-01-01""",null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46958""","""1022""",null,null,""",1,""","""73""","""0""","""9.2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545215.850000,184062.010000 54…",27700,"""POLYGON ((545215.850000 184062…","""Polygon""","""1892756""","""239944""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""","""2016-09-20""","""2199-01-01""",null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46959""","""1022""",null,null,""",1,""","""73""","""0""","""9.2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545262.200000,184035.490000 54…",27700,"""POLYGON ((545262.200000 184035…","""Polygon""","""1892757""","""239945""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""","""2016-09-20""","""2199-01-01""",null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46960""","""1022""",null,null,""",1,""","""73""","""0""","""9.2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [6]:
def correct_dtypes(df:pl.DataFrame) -> pl.DataFrame:
    """
    this function takes a polars DataFrame and corrects the data types
    it converts the date_from and date_to columns to datetime format

    """
    return df.with_columns(
    pl.col("date_from").str.strptime(pl.Date, "%Y-%m-%d"),
    pl.col("date_to").str.strptime(pl.Date, "%Y-%m-%d")
    )
raw_df= correct_dtypes(df_traffweb)
raw_df.head()

layer,borough,boundedBy,msGeometry,geom_raw,geom_srid,wkt,geom_type,ogc_fid,item_ref,order_ref,order_type,street_name,side_of_road,locality,district,ordstart,ordfinish,ordlocation,schedule,date_from,date_to,pre_description,times_of_enforcement,post_description,sp_filename,sp_blockname,entry_type,restriction,pm_id,order_id,order_doc,blockimagefile,ms_grid,type_ref,no_of_spaces,length,tariff_code,tariff,pbp_code,pbp_tariff,zone_code,order_type_ln,side_of_road_ln,restriction_ln,tariff1,tariff2,poly_id,polyname,poly_info,zonecode,charges_doc
str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,date,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,null,str,null,str,null,str,null,str,str,str,str,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545286.070000,183948.350000 54…",27700,"""POLYGON ((545286.070000 183948…","""Polygon""","""1892753""","""239941""","""Consolidation 2016""","""Permit Parking Area""","""Lancaster Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46956""","""1022""",null,null,""",1,""","""73""","""0""","""9.2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545246.110000,184107.900000 54…",27700,"""POLYGON ((545246.110000 184107…","""Polygon""","""1892754""","""239942""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46957""","""1022""",null,null,""",1,""","""73""","""0""","""0""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545235.900000,184109.950000 54…",27700,"""POLYGON ((545235.900000 184109…","""Polygon""","""1892755""","""239943""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46958""","""1022""",null,null,""",1,""","""73""","""0""","""9.2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545215.850000,184062.010000 54…",27700,"""POLYGON ((545215.850000 184062…","""Polygon""","""1892756""","""239944""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46959""","""1022""",null,null,""",1,""","""73""","""0""","""9.2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ordersr""","""barking_dagenham""",""" """,""" ""","""545262.200000,184035.490000 54…",27700,"""POLYGON ((545262.200000 184035…","""Polygon""","""1892757""","""239945""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,null,null,null,null,null,"""Region""","""Resident Permit Parking Area Z…","""46960""","""1022""",null,null,""",1,""","""73""","""0""","""9.2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [13]:
print(raw_df.columns)

['layer', 'borough', 'boundedBy', 'msGeometry', 'geom_raw', 'geom_srid', 'wkt', 'geom_type', 'ogc_fid', 'item_ref', 'order_ref', 'order_type', 'street_name', 'side_of_road', 'locality', 'district', 'ordstart', 'ordfinish', 'ordlocation', 'schedule', 'date_from', 'date_to', 'pre_description', 'times_of_enforcement', 'post_description', 'sp_filename', 'sp_blockname', 'entry_type', 'restriction', 'pm_id', 'order_id', 'order_doc', 'blockimagefile', 'ms_grid', 'type_ref', 'no_of_spaces', 'length', 'tariff_code', 'tariff', 'pbp_code', 'pbp_tariff', 'zone_code', 'order_type_ln', 'side_of_road_ln', 'restriction_ln', 'tariff1', 'tariff2', 'poly_id', 'polyname', 'poly_info', 'zonecode', 'charges_doc']


In [19]:
restructure=['pm_id',
'layer', 
 'borough', 
 'boundedBy', 
 'msGeometry',  
 'item_ref', 
 'order_ref', 
 'order_type', 
 'street_name', 
 'side_of_road', 
 'locality', 
 'district', 
 'ordstart', 
 'ordfinish', 
 'ordlocation', 
 'schedule', 
 'date_from', 
 'date_to', 
 'pre_description', 
 'times_of_enforcement', 
 'post_description', 
 'sp_filename', 
 'sp_blockname', 
 'entry_type', 
 'restriction', 
 'order_id', 
 'order_doc', 
 'blockimagefile', 
 'ms_grid', 
 'type_ref', 
 'no_of_spaces', 
 'length', 
 'tariff_code', 
 'tariff', 
 'pbp_code', 
 'pbp_tariff', 
 'zone_code', 
 'order_type_ln', 
 'side_of_road_ln', 
 'restriction_ln', 
 'tariff1', 
 'tariff2', 
 'poly_id', 
 'polyname', 
 'poly_info', 
 'zonecode', 
 'charges_doc',
 'geom_raw', 
 'geom_srid',  
 'geom_type',
 'wkt']

In [32]:
raw_restructure= raw_df.select(restructure)
display(raw_restructure.filter(pl.col("poly_info").is_not_null()).head())
"""
pm_id=> primary key for the traffic order record, unique identifier for each order, int
layer => the layer name from which the order was extracted, string
borough => the borough from which the order was extracted, string
boundedBy => the bounding box of the feature, string
msGeometry => the raw geometry data in GML format, string
item_ref => reference number for the item,int
order_ref => reference number for the order, str
order_type => type of order, string
street_name => name of the street where the order applies, string
side_of_road => side of the road where the order applies, string
locality => locality of the order, string
district => district of the order, string
ordstart => string description of the order where the restriction starts, string
ordfinish => string description of the order where the restriction finishes, string
ordlocation => string description of the location of the order, string
schedule => string description of the schedule of the order, string
date_from => the date when the order starts, datetime
date_to => the date when the order ends, datetime
pre_description => the description of the order before the schedule, string
times_of_enforcement => string description of the times when the order is enforced, string
post_description => the description of the order after the schedule, string
sp_filename => filename of the special order document, string
sp_blockname => block name for the special order, string
entry_type => type of entry for the order, string
restriction => string description of the restriction imposed by the order, string
order_id => unique identifier for the order, string
order_doc => document reference for the order, string
blockimagefile => filename of the block image for the order, string
ms_grid => grid reference for the order, string
type_ref => reference for the type of order, string
no_of_spaces => number of parking spaces affected by the order, int
length => length of the restriction in meters, float
tariff_code => code for the tariff applicable to the order, string
tariff => description of the tariff applicable to the order, string
pbp_code => code for the pay by phone tariff, string
pbp_tariff => description of the pay by phone tariff, string
zone_code => code for the zone where the order applies, string
order_type_ln => long name for the type of order, string
side_of_road_ln => long name for the side of the road, string
restriction_ln => long name for the restriction, string
tariff1 => first tariff applicable to the order, string
tariff2 => second tariff applicable to the order, string
poly_id => identifier for the polygon associated with the order, string
polyname => name of the polygon associated with the order, string
poly_info => additional information about the polygon, string
zonecode => code for the zone where the order applies, string
charges_doc => document reference for the charges associated with the order, string
geom_raw => the raw geometry data extracted from the GML, string
geom_srid => the spatial reference identifier for the geometry, int
geom_type => the type of geometry (Point, LineString, Polygon), string
wkt => the geometry in Well-Known Text format, string
"""

pm_id,layer,borough,boundedBy,msGeometry,item_ref,order_ref,order_type,street_name,side_of_road,locality,district,ordstart,ordfinish,ordlocation,schedule,date_from,date_to,pre_description,times_of_enforcement,post_description,sp_filename,sp_blockname,entry_type,restriction,order_id,order_doc,blockimagefile,ms_grid,type_ref,no_of_spaces,length,tariff_code,tariff,pbp_code,pbp_tariff,zone_code,order_type_ln,side_of_road_ln,restriction_ln,tariff1,tariff2,poly_id,polyname,poly_info,zonecode,charges_doc,geom_raw,geom_srid,geom_type,wkt
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,date,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,null,str,null,str,null,str,null,str,str,str,str,null,str,i64,str,str
null,"""twpzones""","""camden""",""" """,""" """,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""51""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""1""","""CA-B Belsize""","""CAB""","""CA-B""",null,"""526983.550000,185229.540000 52…",27700,"""Polygon""","""POLYGON ((526983.550000 185229…"
null,"""twpzones""","""camden""",""" """,""" """,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""52""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""3""","""CA-C (Single Yellow lines)""","""CAC""","""CA-C""",null,"""529538.070000,181539.480000 52…",27700,"""Polygon""","""POLYGON ((529538.070000 181539…"
null,"""twpzones""","""camden""",""" """,""" """,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""53""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""4""","""CA-D Kings Cross Area""","""CAD""","""CA-D""",null,"""530369.920000,181796.910000 53…",27700,"""Polygon""","""POLYGON ((530369.920000 181796…"
null,"""twpzones""","""camden""",""" """,""" """,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""54""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""5""","""CA-D/E""","""CAD""","""CA-D/E""",null,"""529407.900000,182392.080000 52…",27700,"""Polygon""","""POLYGON ((529407.900000 182392…"
null,"""twpzones""","""camden""",""" """,""" """,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""55""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""6""","""CA-E Bloomsbury Area""","""CAE""","""CA-E""",null,"""528931.980000,182185.860000 52…",27700,"""Polygon""","""POLYGON ((528931.980000 182185…"


'\npm_id=> primary key for the traffic order record, unique identifier for each order, int\nlayer => the layer name from which the order was extracted, string\nborough => the borough from which the order was extracted, string\nboundedBy => the bounding box of the feature, string\nmsGeometry => the raw geometry data in GML format, string\nitem_ref => reference number for the item,int\norder_ref => reference number for the order, str\norder_type => type of order, string\nstreet_name => name of the street where the order applies, string\nside_of_road => side of the road where the order applies, string\nlocality => locality of the order, string\ndistrict => district of the order, string\nordstart => string description of the order where the restriction starts, string\nordfinish => string description of the order where the restriction finishes, string\nordlocation => string description of the location of the order, string\nschedule => string description of the schedule of the order, string\

In [ ]:
def get_max_lengths(df: pl.DataFrame, cols: list[str]) -> dict[str, int]:
    max_lengths = {}
    for col in cols:
        if col in df.columns:
            max_length = df.select(pl.col(col).cast(pl.Utf8).str.len_chars().max()).item()
            max_lengths[col] = max_length
        else:
            max_lengths[col] = None
    return max_lengths
cols_to_check = ['pm_id', 'layer', 'borough', 'boundedBy', 'msGeometry', 'item_ref', 'order_ref', 'order_type', 'street_name', 'side_of_road', 'locality', 'district', 'ordstart', 'ordfinish', 'ordlocation', 'schedule', 'pre_description', 'times_of_enforcement', 'post_description', 'sp_filename', 'sp_blockname', 'entry_type', 'restriction', 'order_id', 'order_doc', 'blockimagefile', 'ms_grid', 'type_ref', 'tariff_code', 'tariff', 'pbp_code', 'pbp_tariff', 'zone_code', 'order_type_ln', 'side_of_road_ln', 'restriction_ln', 'tariff1', 'tariff2', 'poly_id', 'polyname', 'poly_info','zonecode','charges_doc', 'geom_raw', 'wkt']
max_lengths = get_max_lengths(raw_df, cols_to_check)
display(max_lengths)

{'pm_id': 6,
 'layer': 8,
 'borough': 16,
 'boundedBy': 10,
 'msGeometry': 9,
 'item_ref': 8,
 'order_ref': 38,
 'order_type': 50,
 'street_name': 104,
 'side_of_road': 35,
 'locality': 30,
 'district': 29,
 'ordstart': 254,
 'ordfinish': 237,
 'ordlocation': 254,
 'schedule': 9,
 'pre_description': 100,
 'times_of_enforcement': 63,
 'post_description': 90,
 'sp_filename': 8,
 'sp_blockname': 6,
 'entry_type': 6,
 'restriction': 212,
 'order_id': 4,
 'order_doc': 35,
 'blockimagefile': 10,
 'ms_grid': 7,
 'type_ref': 3,
 'tariff_code': 17,
 'tariff': 254,
 'pbp_code': 6,
 'pbp_tariff': None,
 'zone_code': 20,
 'order_type_ln': None,
 'side_of_road_ln': 35,
 'restriction_ln': None,
 'tariff1': 172,
 'tariff2': None,
 'poly_id': 2,
 'polyname': 40,
 'poly_info': 35,
 'zonecode': 20,
 'charges_doc': None,
 'geom_raw': 240603,
 'wkt': 249207}

In [11]:
raw_restructure= raw_df.select(
    "pm_id",
    "layer",
    "borough",
    "date_from",
    "date_to",
    "wkt",
    "geom_type"
)

In [12]:
display(raw_restructure.head())

pm_id,layer,borough,date_from,date_to,wkt,geom_type
str,str,str,date,date,str,str
"""46956""","""ordersr""","""barking_dagenham""",2016-09-20,2199-01-01,"""POLYGON ((545286.070000 183948…","""Polygon"""
"""46957""","""ordersr""","""barking_dagenham""",2016-09-20,2199-01-01,"""POLYGON ((545246.110000 184107…","""Polygon"""
"""46958""","""ordersr""","""barking_dagenham""",2016-09-20,2199-01-01,"""POLYGON ((545235.900000 184109…","""Polygon"""
"""46959""","""ordersr""","""barking_dagenham""",2016-09-20,2199-01-01,"""POLYGON ((545215.850000 184062…","""Polygon"""
"""46960""","""ordersr""","""barking_dagenham""",2016-09-20,2199-01-01,"""POLYGON ((545262.200000 184035…","""Polygon"""


In [9]:
# convert wkt to geom using shapely
import geopandas as gpd
import shapely.wkt as shapely_wkt
pdf = raw_df.to_pandas()
pdf["geometry"] = pdf["wkt"].apply(
    lambda x: shapely_wkt.loads(x) if x else None
)

gdf = gpd.GeoDataFrame(pdf, geometry="geometry", crs="EPSG:27700")

In [10]:
gdf.head()

,layer,borough,boundedBy,msGeometry,geom_raw,geom_srid,wkt,geom_type,ogc_fid,item_ref,...,side_of_road_ln,restriction_ln,tariff1,tariff2,poly_id,polyname,poly_info,zonecode,charges_doc,geometry
0,ordersr,barking_dagenham,\n \t,\n,"545286.070000,183948.350000 545279.970000,1839...",27700,"POLYGON ((545286.070000 183948.350000, 545279....",Polygon,1892753,239941,...,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,"POLYGON ((545286.07 183948.35, 545279.97 18398..."
1,ordersr,barking_dagenham,\n \t,\n,"545246.110000,184107.900000 545245.200000,1841...",27700,"POLYGON ((545246.110000 184107.900000, 545245....",Polygon,1892754,239942,...,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,"POLYGON ((545246.11 184107.9, 545245.2 184113...."
2,ordersr,barking_dagenham,\n \t,\n,"545235.900000,184109.950000 545238.400000,1841...",27700,"POLYGON ((545235.900000 184109.950000, 545238....",Polygon,1892755,239943,...,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,"POLYGON ((545235.9 184109.95, 545238.4 184103...."
3,ordersr,barking_dagenham,\n \t,\n,"545215.850000,184062.010000 545260.380000,1840...",27700,"POLYGON ((545215.850000 184062.010000, 545260....",Polygon,1892756,239944,...,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,"POLYGON ((545215.85 184062.01, 545260.38 18404..."
4,ordersr,barking_dagenham,\n \t,\n,"545262.200000,184035.490000 545262.880000,1840...",27700,"POLYGON ((545262.200000 184035.490000, 545262....",Polygon,1892757,239945,...,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,"POLYGON ((545262.2 184035.49, 545262.88 184039..."


In [8]:
df_traffweb.write_csv("../data/traffweb_8_boroughs.csv")

* Ingest RAW

In [350]:
"""
code to ingest raw into postgis

remove fid as PK is pm_id

turn date strings into date format 

this is all bronze layer
"""

'\ncode to ingest raw into postgis\n\nremove fid as PK is pm_id\n\nturn date strings into date format \n\nthis is all bronze layer\n'

* Data cleansing and some enrichment 

In [351]:
# TODO: clean up the data for silver layer
# remove cols that have no data or are not relevant to the traffic orders
# restruture the data 
# parse the restriction text to extract structured information about the restrictions
# Ingest the data into postgis bronze layer and silver layer
# 
# 
# 

In [353]:
display(df)

shape: (0, 0)
┌┐
╞╡
└┘

In [9]:
"""
turning date strings into date format for postgis ingestion

"""

raw_df=df_traffweb.with_columns(
    pl.col("date_from").str.strptime(pl.Date, "%Y-%m-%d"),
    pl.col("date_to").str.strptime(pl.Date, "%Y-%m-%d")
)

In [10]:
"""

drop cols that have no data or not relevant to the traffic orders for silver layer

"""

drop_cols=[
    'pre_description', 
    'times_of_enforcement', 
    'post_description', 
    'sp_filename', 
    'sp_blockname', 
    'blockimagefile', 
    'ms_grid', 
    'tariff_code', 
    'tariff', 
    'pbp_code', 
    'pbp_tariff', 
    'zone_code', 
    'order_type_ln', 
    'restriction_ln', 
    'tariff1', 
    'tariff2',
    "msGeometry",
    "boundedBy",
]
clean_df=raw_df.drop(drop_cols)


In [11]:
from datetime import date

today = date.today()

df_silver = clean_df.filter(
    # keep where date_to is null (permanent) or date_to >= today
    (pl.col("date_to").is_null()) | (pl.col("date_to") >= today)
).filter(
    # keep where date_from is null or date_from <= today
    (pl.col("date_from").is_null()) | (pl.col("date_from") <= today)
)
print("Records before date filter:", len(clean_df))
print(f"Records after date filter: {len(df_silver)}")

Records before date filter: 217426
Records after date filter: 215742


In [12]:
df_silver = df_silver.filter(
    ~pl.col("layer").str.starts_with("morders")
)

In [13]:
def categorise_restriction(order_type: str, restriction: str) -> str:
    if order_type is None:
        return "unknown"
    
    ot = order_type.lower()
    
    if "clearway" in ot:
        return "clearway"
    if "loading" in ot:
        return "loading_only"
    if "disabled" in ot:
        return "disabled_bay"
    if "ambulance" in ot:
        return "disabled_bay"        # emergency vehicle bay
    if "permit" in ot:
        return "permit_only"
    if "resident" in ot:
        return "permit_only"
    if "shared use" in ot:
        return "shared_use"          # permit OR pay depending on time
    if "pay" in ot or "display" in ot:
        return "pay_display"
    if "electric" in ot or "ev" in ot or "charging" in ot:
        return "ev_charging"
    if "no waiting" in ot:
        return "no_waiting"
    if "red route" in ot or "tfl" in ot:
        return "red_route"
    if "cycle hire" in ot:
        return "cycle_hire"
    if "event" in ot or "stadium" in ot or "match" in ot:
        return "event_day"           # dynamic restriction — flag separately
    if "off-street car park" in ot:
        return "off_street_car_park"
    if "motorcycle parking" in ot:
        return "motorcycle_parking"
    if "school keep clear" in ot:
        return "school_keep_clear"
    
    if "streetcar parking" in ot:
        return "streetcar_parking"
    if "limited waiting" in ot:
        return "limited_waiting"
    
    return "unknown"
df_silver = df_silver.with_columns(
    pl.struct(["order_type", "restriction"])
    .map_elements(
        lambda x: categorise_restriction(x["order_type"], x["restriction"]),
        return_dtype=pl.Utf8
    )
    .alias("restriction_category")
)

In [14]:
# any wkt that doesnt start with expected types
bad_wkt = df_silver.filter(
    ~pl.col("wkt").str.starts_with("POINT") &
    ~pl.col("wkt").str.starts_with("LINESTRING") &
    ~pl.col("wkt").str.starts_with("POLYGON")
)
print(f"Bad WKT rows: {len(bad_wkt)}")

Bad WKT rows: 0


In [15]:
import re
from datetime import time

def parse_time(time_str: str) -> str | None:
    """Convert time string like '9.30am', '6pm', '11am' to HH:MM string"""
    if not time_str:
        return None
    
    time_str = time_str.strip().lower()
    
    match = re.match(r"(\d{1,2})(?:\.(\d{2}))?([ap]m)", time_str)
    if not match:
        return None
    
    hour = int(match.group(1))
    minute = int(match.group(2)) if match.group(2) else 0
    period = match.group(3)
    
    if period == "pm" and hour != 12:
        hour += 12
    elif period == "am" and hour == 12:
        hour = 0
    
    # return as HH:MM string not time object
    return f"{hour:02d}:{minute:02d}"


def parse_days(days_str: str) -> list[str]:
    """Convert day range string like 'Mon-Fri', 'Mon-Sat', 'Sat-Sun' to list of days"""
    all_days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    
    if not days_str:
        return []
    
    days_str = days_str.strip()
    result = set()
    
    # handle ranges like Mon-Fri, Mon-Sat, Sat-Sun
    range_match = re.findall(r"(Mon|Tue|Wed|Thu|Fri|Sat|Sun)-(Mon|Tue|Wed|Thu|Fri|Sat|Sun)", days_str)
    for start, end in range_match:
        start_idx = all_days.index(start)
        end_idx = all_days.index(end)
        for i in range(start_idx, end_idx + 1):
            result.add(all_days[i])
    
    # handle individual days
    individual = re.findall(r"\b(Mon|Tue|Wed|Thu|Fri|Sat|Sun)\b", days_str)
    for day in individual:
        result.add(day)
    
    # return in week order
    return [d for d in all_days if d in result]


def parse_restriction(restriction: str) -> dict:
    """
    Parse restriction text into structured time fields.
    
    Handles patterns:
    - No waiting at any time
    - No waiting Mon-Fri 9.30am-6pm
    - No waiting Mon-Fri 9.30am-6pm and Sat 9.30am-12.30pm
    - No waiting Mon-Fri 9.30am-12.30pm and 4.30pm-6.30pm and Sat 9.30am-12.30pm
    - No waiting 7am-11pm on event days
    - Resident Permit Holders Mon-Fri 9.30am-6pm Except Christmas Day...
    """
    result = {
        "is_any_time":    False,
        "has_exceptions": False,
        "is_event_day":   False,
        "overnight":      False,
        "needs_review":   False,
        "days_of_week":   None,
        "start_time":     None,
        "end_time":       None,
        "sat_start_time": None,
        "sat_end_time":   None,
        "time_window_2_start": None,  # for split windows like 9-12 and 4-6
        "time_window_2_end":   None,
    }
    
    if not restriction:
        result["needs_review"] = True
        return result
    
    r = restriction.strip()
    
    # --- at any time ---
    if re.search(r"at any time", r, re.IGNORECASE):
        result["is_any_time"] = True
        return result
    
    # --- exceptions ---
    if re.search(r"bank holiday|christmas day|good friday", r, re.IGNORECASE):
        result["has_exceptions"] = True
    
    # --- event days ---
    if re.search(r"event day|match day|brentford fc|stadium", r, re.IGNORECASE):
        result["is_event_day"] = True
        # still try to parse times for event day restrictions
        time_match = re.search(r"(\d{1,2}(?:\.\d{2})?[ap]m)-(\d{1,2}(?:\.\d{2})?[ap]m)", r)
        if time_match:
            result["start_time"] = parse_time(time_match.group(1))
            result["end_time"] = parse_time(time_match.group(2))
        return result
    
    # --- parse day ranges ---
    # look for day pattern before the times
    day_match = re.search(
        r"((?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)(?:-(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun))?)",
        r
    )
    if day_match:
        result["days_of_week"] = parse_days(day_match.group(1))
    
    # --- find all time windows ---
    # pattern: time-time (e.g. 9.30am-6pm)
    time_pattern = r"(\d{1,2}(?:\.\d{2})?[ap]m)-(\d{1,2}(?:\.\d{2})?[ap]m)"
    
    # split on "and Sat" to handle Saturday separately
    sat_split = re.split(r"\band\s+Sat\b", r, flags=re.IGNORECASE)
    
    main_part = sat_split[0]
    sat_part = sat_split[1] if len(sat_split) > 1 else None
    
    # find time windows in main part
    main_windows = re.findall(time_pattern, main_part)
    
    if len(main_windows) >= 1:
        result["start_time"] = parse_time(main_windows[0][0])
        result["end_time"] = parse_time(main_windows[0][1])
    
    if len(main_windows) >= 2:
        # split window like 9.30am-12.30pm and 4.30pm-6.30pm
        result["time_window_2_start"] = parse_time(main_windows[1][0])
        result["time_window_2_end"] = parse_time(main_windows[1][1])
    
    # split on "and Sat" — use capturing group to preserve Sat
    sat_split = re.split(r"\band\s+(Sat(?:-Sun)?)\b", r, flags=re.IGNORECASE)

    main_part = sat_split[0]

    if len(sat_split) > 2:
        sat_part = sat_split[1] + sat_split[2]  # "Sat-Sun -2pm-4pm"
    elif len(sat_split) > 1:
        sat_part = "Sat " + sat_split[1]
    else:
        sat_part = None

    # find time windows in main part
    main_windows = re.findall(time_pattern, main_part)

    if len(main_windows) >= 1:
        result["start_time"] = parse_time(main_windows[0][0])
        result["end_time"] = parse_time(main_windows[0][1])

    if len(main_windows) >= 2:
        result["time_window_2_start"] = parse_time(main_windows[1][0])
        result["time_window_2_end"] = parse_time(main_windows[1][1])

    if sat_part:
        sat_windows = re.findall(time_pattern, sat_part)
        if sat_windows:
            result["sat_start_time"] = parse_time(sat_windows[0][0])
            result["sat_end_time"] = parse_time(sat_windows[0][1])

        all_days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        existing = set(result["days_of_week"] or [])
        existing.add("Sat")
        if re.search(r"\bSun\b", sat_part, re.IGNORECASE):
            existing.add("Sun")
        result["days_of_week"] = [d for d in all_days if d in existing]
        
    # --- overnight check ---
    if result["start_time"] and result["end_time"]:
        start_h, start_m = map(int, result["start_time"].split(":"))
        end_h, end_m = map(int, result["end_time"].split(":"))
        if (end_h * 60 + end_m) < (start_h * 60 + start_m):
            result["overnight"] = True
    
    # --- needs review if no times parsed and not AAT or event ---
    if (not result["is_any_time"] and 
        not result["is_event_day"] and 
        result["start_time"] is None and
        not re.search(r"cycle hire|disabled|ambulance", r, re.IGNORECASE)):
        result["needs_review"] = True
    
    return result

In [16]:
df_silver = df_silver.with_columns(
    pl.col("no_of_spaces").cast(pl.Int64, strict=False)
)

In [17]:
existing_to_drop = [c for c in ["is_any_time"] if c in df_silver.columns]
df_silver = df_silver.drop(existing_to_drop)

df_silver = df_silver.with_columns(
    pl.col("restriction").map_elements(
        lambda r: parse_restriction(r) if r else parse_restriction(""),
        return_dtype=pl.Struct({
            "is_any_time":         pl.Boolean,
            "has_exceptions":      pl.Boolean,
            "is_event_day":        pl.Boolean,
            "overnight":           pl.Boolean,
            "needs_review":        pl.Boolean,
            "days_of_week":        pl.List(pl.Utf8),
            "start_time":          pl.Utf8,
            "end_time":            pl.Utf8,
            "sat_start_time":      pl.Utf8,
            "sat_end_time":        pl.Utf8,
            "time_window_2_start": pl.Utf8,
            "time_window_2_end":   pl.Utf8,
        })
    ).alias("parsed")
).unnest("parsed")

In [18]:
result = parse_restriction("No waiting Mon-Fri 9am-11am and 5pm-7pm and Sat-Sun 2pm-4pm")
print(result["days_of_week"])

['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']


In [19]:
test_cases = [
    "No waiting at any time",
    "No waiting Mon-Fri 9.30am-6pm and Sat 9.30am-12.30pm",
    "No waiting Mon-Fri 9.30am-12.30pm and 4.30pm-6.30pm and Sat 9.30am-12.30pm",
    "Resident Permit Holders Mon-Fri 9.30am-6pm Except Christmas Day, Good Friday or a Bank Holiday",
    "No waiting 7am-11pm on event days",
    "No waiting Mon-Fri 9am-11am and 5pm-7pm and Sat-Sun 2pm-4pm",
]

for t in test_cases:
    result = parse_restriction(t)
    print(f"\n{t}")
    # print(f"  days={result['days_of_week']} start={result['start_time']} end={result['end_time']}")
    print(f"  sat_start={result['sat_start_time']} sat_end={result['sat_end_time']}")
    # print(f"  window2={result['time_window_2_start']}-{result['time_window_2_end']}")
    # print(f"  aat={result['is_any_time']} event={result['is_event_day']} exceptions={result['has_exceptions']} overnight={result['overnight']} review={result['needs_review']}")


No waiting at any time
  sat_start=None sat_end=None

No waiting Mon-Fri 9.30am-6pm and Sat 9.30am-12.30pm
  sat_start=09:30 sat_end=12:30

No waiting Mon-Fri 9.30am-12.30pm and 4.30pm-6.30pm and Sat 9.30am-12.30pm
  sat_start=09:30 sat_end=12:30

Resident Permit Holders Mon-Fri 9.30am-6pm Except Christmas Day, Good Friday or a Bank Holiday
  sat_start=None sat_end=None

No waiting 7am-11pm on event days
  sat_start=None sat_end=None

No waiting Mon-Fri 9am-11am and 5pm-7pm and Sat-Sun 2pm-4pm
  sat_start=14:00 sat_end=16:00


In [20]:
test_cases = [
    "No waiting Mon-Fri 9am-11am and 5pm-7pm and Sat-Sun 2pm-4pm",
]

for t in test_cases:
    result = parse_restriction(t)
    print(f"{t}")
    print(f"  days={result['days_of_week']}")
    print(f"  start={result['start_time']} end={result['end_time']}")
    print(f"  window2={result['time_window_2_start']}-{result['time_window_2_end']}")
    print(f"  sat_start={result['sat_start_time']} sat_end={result['sat_end_time']}")

No waiting Mon-Fri 9am-11am and 5pm-7pm and Sat-Sun 2pm-4pm
  days=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
  start=09:00 end=11:00
  window2=17:00-19:00
  sat_start=14:00 sat_end=16:00


In [307]:
print(parse_days("Sat-Sun 2pm-4pm"))

['Sat', 'Sun']


In [36]:
display(df_silver)

layer,borough,geom_raw,geom_srid,wkt,geom_type,ogc_fid,item_ref,order_ref,order_type,street_name,side_of_road,locality,district,ordstart,ordfinish,ordlocation,schedule,date_from,date_to,entry_type,restriction,pm_id,order_id,order_doc,type_ref,no_of_spaces,length,side_of_road_ln,poly_id,polyname,poly_info,zonecode,charges_doc,restriction_category,is_any_time,has_exceptions,is_event_day,overnight,needs_review,days_of_week,start_time,end_time,sat_start_time,sat_end_time,time_window_2_start,time_window_2_end
str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,date,date,str,str,str,str,str,str,i64,str,str,str,str,str,str,null,str,bool,bool,bool,bool,bool,list[str],str,str,str,str,str,str
"""ordersr""","""barking_dagenham""","""545286.070000,183948.350000 54…",27700,"""POLYGON ((545286.070000 183948…","""Polygon""","""1882638""","""239941""","""Consolidation 2016""","""Permit Parking Area""","""Lancaster Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,"""Region""","""Resident Permit Parking Area Z…","""46956""","""1022""",null,"""73""",0,"""9.2""",null,null,null,null,null,null,"""permit_only""",false,false,false,false,false,"[""Mon"", ""Tue"", … ""Fri""]","""08:30""","""17:30""",null,null,null,null
"""ordersr""","""barking_dagenham""","""545246.110000,184107.900000 54…",27700,"""POLYGON ((545246.110000 184107…","""Polygon""","""1882639""","""239942""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,"""Region""","""Resident Permit Parking Area Z…","""46957""","""1022""",null,"""73""",0,"""0""",null,null,null,null,null,null,"""permit_only""",false,false,false,false,false,"[""Mon"", ""Tue"", … ""Fri""]","""08:30""","""17:30""",null,null,null,null
"""ordersr""","""barking_dagenham""","""545235.900000,184109.950000 54…",27700,"""POLYGON ((545235.900000 184109…","""Polygon""","""1882640""","""239943""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,"""Region""","""Resident Permit Parking Area Z…","""46958""","""1022""",null,"""73""",0,"""9.2""",null,null,null,null,null,null,"""permit_only""",false,false,false,false,false,"[""Mon"", ""Tue"", … ""Fri""]","""08:30""","""17:30""",null,null,null,null
"""ordersr""","""barking_dagenham""","""545215.850000,184062.010000 54…",27700,"""POLYGON ((545215.850000 184062…","""Polygon""","""1882641""","""239944""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,"""Region""","""Resident Permit Parking Area Z…","""46959""","""1022""",null,"""73""",0,"""9.2""",null,null,null,null,null,null,"""permit_only""",false,false,false,false,false,"[""Mon"", ""Tue"", … ""Fri""]","""08:30""","""17:30""",null,null,null,null
"""ordersr""","""barking_dagenham""","""545262.200000,184035.490000 54…",27700,"""POLYGON ((545262.200000 184035…","""Polygon""","""1882642""","""239945""","""Consolidation 2016""","""Permit Parking Area""","""Coniston Avenue""",null,null,"""Barking""",null,null,null,"""X640""",2016-09-20,2199-01-01,"""Region""","""Resident Permit Parking Area Z…","""46960""","""1022""",null,"""73""",0,"""9.2""",null,null,null,null,null,null,"""permit_only""",false,false,false,false,false,"[""Mon"", ""Tue"", … ""Fri""]","""08:30""","""17:30""",null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ordersl""","""redbridge""","""544724.890000,186187.290000 54…",27700,"""LINESTRING (544724.890000 1861…","""LineString""","""21338079""","""209674""","""LBR. 4 (2024)""","""Electric Vehicles Only""","""GOLFE ROAD""","""north-west""","""ILFORD""","""REDBRIDGE""",null,null,null,"""X1493""",2024-02-09,2199-01-01,"""line""","""Electric Vehicle Recharging Po…","""18638""","""1048""",null,"""68""",2,"""11.5""","""north-west""",nu

In [2]:
"""create a staging dataframe with the relevant cols for postgis ingestion for silver layer"""


stg_df=df_silver.select([
    "pm_id"
    ,"date_from"
    ,"date_to"
    ,"street_name"
    ,"restriction"
    ,"side_of_road"
    ,'restriction_category' 
    ,'is_any_time'
    ,'has_exceptions' 
    ,'is_event_day' 
    ,'overnight' 
    ,'needs_review' 
    ,'days_of_week' 
    ,'start_time' 
    ,'end_time' 
    ,'sat_start_time' 
    ,'sat_end_time' 
    ,'time_window_2_start' 
    ,'time_window_2_end'
    ,"locality"
    ,"district"
    ,"ordlocation"
    ,"ordstart"
    ,"ordfinish"
    ,"schedule"
    ,"zonecode"
    ,"charges_doc"
    ,"entry_type"
    ,"length"
    ,"side_of_road_ln"
    ,"no_of_spaces"
    ,"order_id"
    ,"order_type"
    ,"order_ref"
    ,"order_doc"
    ,"type_ref"
    ,"borough"
    ,"layer"
    ,"wkt"
])



stg_df.head()



# ['layer', 
#  'borough', 
#  'geom_raw', 
#  'geom_srid', 
#  'wkt', 
#  'geom_type', 
#  'ogc_fid', 
#  'item_ref', 
#  'order_ref', 
#  'order_type', 
#  'street_name', 
#  'side_of_road', 
#  'locality', 
#  'district', 
#  'ordstart', 
#  'ordfinish', 
#  'ordlocation', 
#  'schedule', 
#  'date_from', 
#  'date_to', 
#  'entry_type', 
#  'restriction', 
#  'pm_id', 
#  'order_id', 
#  'order_doc', 
#  'type_ref', 
#  'no_of_spaces', 
#  'length', 
#  'side_of_road_ln', 
#  'poly_id', 
#  'polyname', 
#  'poly_info', 
#  'zonecode', 
#  'charges_doc', 
#  'restriction_category', 
#  'is_any_time', 
#  'has_exceptions', 
#  'is_event_day', 
#  'overnight', 
#  'needs_review', 
#  'days_of_week', 
#  'start_time', 
#  'end_time', 
#  'sat_start_time', 
#  'sat_end_time', 
#  'time_window_2_start', 
#  'time_window_2_end']

NameError: name 'df_silver' is not defined

In [50]:
# simulate what your API will do for The Drive Feltham

feltham_drive = df_silver.filter(
    (pl.col("street_name") == "THE DRIVE") &
    (pl.col("district") == "FELTHAM")
)

print("All restrictions on The Drive Feltham:")
print(feltham_drive.select([
    "restriction", "ordstart", "ordfinish", "side_of_road"
]))

# detect junction only pattern
# signal 1 — ordfinish contains "10.0 metres" or "10 metres"
# signal 2 — ordstart contains "kerbline" — means it starts at a junction

is_junction_only = (
    feltham_drive
    .filter(
        pl.col("ordfinish").str.contains("10.0 metres|10 metres") &
        pl.col("ordstart").str.contains("kerbline")
    )
    .height == feltham_drive.height  # ALL rows match the pattern
)

print(f"\nAll restrictions are junction only: {is_junction_only}")

if is_junction_only:
    print("Status: FREE")
    print("Note: Double yellows at junctions only — remainder of street unrestricted")
else:
    print("Status: RESTRICTED")

All restrictions on The Drive Feltham:
shape: (4, 4)
┌────────────────────────┬─────────────────────┬────────────────────────────────────┬──────────────┐
│ restriction            ┆ ordstart            ┆ ordfinish                          ┆ side_of_road │
│ ---                    ┆ ---                 ┆ ---                                ┆ ---          │
│ str                    ┆ str                 ┆ str                                ┆ str          │
╞════════════════════════╪═════════════════════╪════════════════════════════════════╪══════════════╡
│ No waiting at any time ┆ from the north-west ┆ to a point 10.0 metres north-w…    ┆ both         │
│                        ┆ kerbline o…         ┆                                    ┆              │
│ No waiting at any time ┆ from the south-east ┆ to a point 10.0 metres south-e…    ┆ both         │
│                        ┆ kerbline o…         ┆                                    ┆              │
│ No waiting at any time ┆ from the so

In [1]:
stg_df.head()

NameError: name 'stg_df' is not defined

In [196]:
import os
for file in os.listdir("../data/"):
    if file.endswith(".csv"):
        os.remove(os.path.join("../data/", file))

In [195]:
print(len(df_silver))
print(len(clean_df))
print(len(df))

24872
25992
25992
